# DyTopo — τ (Tau) Ablation Study

**Paper:** Lu et al. 2026, *DyTopo: Dynamic Topology Optimization for Multi-Agent Reasoning* — arXiv:2602.06039  
**Target runtime:** Google Colab G4 (NVIDIA RTX PRO 6000, 96 GB VRAM)

This notebook reproduces the τ ablation from **Section 5.4 / Table 3** and augments it with structural density measurements from a mock sweep.

| Metric | τ* | Score |
|---|---|---|
| APPS-Competition | **0.3** | 49.81 % |
| Omni-Math | **0.4** | 52.86 % |

**Two failure modes:**
- τ too **low** → over-dense graph → irrelevant message traffic, degraded context utilisation
- τ too **high** → over-sparse graph → prevents useful information flow

In [ ]:
# Cell 1 — GPU check
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies
!pip install sentence-transformers httpx numpy matplotlib pandas nest_asyncio -q

In [ ]:
# Cell 3 — Download dytopo_engine.py and apply nest_asyncio
import os, sys, urllib.request

ENGINE_URL = "https://raw.githubusercontent.com/51103220/realtime-bus-tracking/main/big-data/api/dytopo_engine.py"
ENGINE_PATH = "/content/dytopo_engine.py"

def _write_fallback_engine():
    """Minimal mock engine so the notebook works even without network."""
    stub = '''
import asyncio, random
from dataclasses import dataclass, field
from typing import List, Optional, AsyncGenerator

@dataclass
class DyTopoConfig:
    task_type: str = "code"
    n_agents: int = 4
    tau: float = 0.3
    max_rounds: int = 3
    mock_mode: bool = True

@dataclass
class RoundResult:
    round: int
    agents: List[str]
    adjacency: List[List[int]]
    messages: List[str] = field(default_factory=list)

def _mock_adjacency(n_agents: int, tau: float, rnd: int) -> List[List[int]]:
    """Generate a plausible mock adjacency matrix for a given tau and round."""
    random.seed(int(tau * 1000) + rnd * 17)
    # Edge probability decreases as tau increases (higher threshold = fewer edges)
    # Round 0 (exploration) is always denser
    base_p = max(0.0, 1.0 - tau * 1.1 + (0.15 if rnd == 0 else 0.0))
    A = [[0] * n_agents for _ in range(n_agents)]
    for i in range(n_agents):
        for j in range(n_agents):
            if i != j and random.random() < base_p:
                A[i][j] = 1
    return A

async def run_dytopo(problem: str, cfg: DyTopoConfig) -> AsyncGenerator[RoundResult, None]:
    agents = [f"agent_{i}" for i in range(cfg.n_agents)]
    for rnd in range(cfg.max_rounds):
        await asyncio.sleep(0)  # yield control
        A = _mock_adjacency(cfg.n_agents, cfg.tau, rnd)
        yield RoundResult(round=rnd, agents=agents, adjacency=A)

def build_adjacency(embeddings, tau):
    import numpy as np
    n = len(embeddings)
    A = np.zeros((n, n), dtype=int)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / (norms + 1e-9)
    sim = normed @ normed.T
    for i in range(n):
        for j in range(n):
            if i != j and sim[i, j] >= tau:
                A[i, j] = 1
    return A

def encode(texts):
    import numpy as np
    random.seed(42)
    return np.array([[random.gauss(0, 1) for _ in range(384)] for _ in texts])
'''
    with open(ENGINE_PATH, "w") as f:
        f.write(stub)
    print("[fallback] Wrote inline dytopo_engine.py stub to", ENGINE_PATH)

# Attempt download
try:
    urllib.request.urlretrieve(ENGINE_URL, ENGINE_PATH)
    print("[OK] Downloaded dytopo_engine.py from GitHub")
except Exception as e:
    print(f"[WARN] Download failed ({e}); using inline fallback stub.")
    _write_fallback_engine()

# Add /content to path and import
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from dytopo_engine import DyTopoConfig, run_dytopo, build_adjacency, encode  # noqa: E402
print("[OK] dytopo_engine imported")

import nest_asyncio
nest_asyncio.apply()
print("[OK] nest_asyncio applied")

import asyncio, numpy as np, matplotlib.pyplot as plt, pandas as pd
plt.style.use('dark_background')
print("[OK] All imports complete")

In [ ]:
# Cell 4 — Plot paper Table 3: bars + accuracy line for apps-comp and omni-math
TAU_PAPER   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
APPS_COMP   = [43.61, 44.84, 49.81, 47.35, 44.13, 40.86, 40.47, 37.59, 34.51]
OMNI_MATH   = [42.86, 47.14, 50.00, 52.86, 48.57, 42.86, 41.43, 40.00, 38.58]

BG       = "#0a0e1a"
PANEL_BG = "#0f1828"
C_APPS   = "#38bdf8"   # sky-blue
C_MATH   = "#fbbf24"   # amber
C_GRID   = "#1e293b"

x   = np.array(TAU_PAPER)
w   = 0.035          # bar half-width

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL_BG)

# Bars (offset left/right so they don't overlap)
ax.bar(x - w, APPS_COMP, width=w * 1.8, color=C_APPS, alpha=0.45, label='_nolegend_')
ax.bar(x + w, OMNI_MATH, width=w * 1.8, color=C_MATH, alpha=0.45, label='_nolegend_')

# Lines
ax.plot(x, APPS_COMP, 'o-', color=C_APPS, linewidth=2.5, markersize=8,
        label='APPS-Competition (τ*=0.3)')
ax.plot(x, OMNI_MATH, 's-', color=C_MATH, linewidth=2.5, markersize=8,
        label='Omni-Math (τ*=0.4)')

# Optimal markers
ax.axvline(x=0.3, color=C_APPS, linestyle='--', alpha=0.45, linewidth=1.5)
ax.axvline(x=0.4, color=C_MATH, linestyle='--', alpha=0.45, linewidth=1.5)
ax.annotate('49.81 %\n(τ*=0.3)', xy=(0.3, 49.81),
            xytext=(0.32, 50.4), color=C_APPS,
            fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_APPS, lw=1.2))
ax.annotate('52.86 %\n(τ*=0.4)', xy=(0.4, 52.86),
            xytext=(0.42, 53.5), color=C_MATH,
            fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_MATH, lw=1.2))

ax.set_xlabel('Similarity Threshold τ', fontsize=11, color='white')
ax.set_ylabel('Accuracy (%)', fontsize=11, color='white')
ax.set_title('DyTopo — Table 3: τ Ablation (Lu et al. 2026, arXiv:2602.06039)',
             fontsize=11, color='white', pad=12)
ax.tick_params(colors='white')
ax.spines[:].set_color(C_GRID)
ax.legend(facecolor='#1a2235', edgecolor=C_GRID, labelcolor='white', fontsize=9)
ax.grid(axis='y', color=C_GRID, alpha=0.4)
ax.set_xticks(x)
ax.set_ylim(30, 57)

plt.tight_layout()
plt.savefig('/content/dytopo_table3_paper.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /content/dytopo_table3_paper.png')

In [ ]:
# Cell 5 — Structural sweep: run mock DyTopo at tau in [0.05..0.90], measure avg edge density per round
PROBLEM    = "Write a Python function to find the first duplicate element in a list."
TAU_VALUES = [round(t, 2) for t in np.arange(0.05, 0.95, 0.05)]

async def run_tau_sweep(tau: float):
    cfg = DyTopoConfig(
        task_type='code',
        n_agents=4,
        tau=tau,
        max_rounds=3,
        mock_mode=True
    )
    round_densities = []
    async for rr in run_dytopo(PROBLEM, cfg):
        A = np.array(rr.adjacency)
        N = len(rr.agents)
        density = A.sum() / (N * (N - 1)) if N > 1 else 0.0
        round_densities.append((rr.round, float(density)))
    return round_densities

print(f'Sweeping {len(TAU_VALUES)} tau values ...')
print(f'  Range: {TAU_VALUES[0]} → {TAU_VALUES[-1]}')
print()

tau_results = {}
for tau in TAU_VALUES:
    round_data = asyncio.run(run_tau_sweep(tau))
    tau_results[tau] = round_data
    avg = np.mean([d for _, d in round_data])
    bar = '█' * int(avg * 20)
    print(f'  τ={tau:.2f}  avg_density={avg:5.1%}  {bar}')

print()
print(f'[OK] Sweep complete — {len(tau_results)} tau configurations measured')

In [ ]:
# Cell 6 — Two-panel chart:
#   Left:  avg density vs tau bar chart with regime colours (red=over-dense, green=optimal, yellow=sparse)
#   Right: per-round density for selected taus

BG       = "#0a0e1a"
PANEL_BG = "#0f1828"
C_GRID   = "#1e293b"

taus      = np.array(list(tau_results.keys()))
avg_dens  = np.array([np.mean([d for _, d in tau_results[t]]) for t in taus])

def regime_color(d):
    if d > 0.70:
        return '#f87171'   # red   — over-dense
    elif d < 0.10:
        return '#fbbf24'   # amber — too sparse
    else:
        return '#4ade80'   # green — optimal

bar_colors = [regime_color(d) for d in avg_dens]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor(BG)
fig.suptitle('τ Ablation — Structural Effect on Graph Sparsity (mock DyTopo)',
             fontsize=12, color='white', fontweight='bold', y=1.01)

# ── Left panel ─────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(PANEL_BG)

ax.bar(taus, avg_dens, width=0.04, color=bar_colors, alpha=0.85, zorder=3)
ax.plot(taus, avg_dens, 'o-', color='white', linewidth=1.4, markersize=4,
        alpha=0.55, zorder=4)

# Regime bands
ax.axhspan(0.10, 0.70, alpha=0.07, color='#4ade80', label='Optimal band (10–70%)')
ax.axhspan(0.70, 1.00, alpha=0.07, color='#f87171')
ax.axhspan(0.00, 0.10, alpha=0.07, color='#fbbf24')

# Optimal tau lines from paper
ax.axvline(x=0.3, color='#38bdf8', linestyle='--', linewidth=1.6, alpha=0.75,
           label='τ*=0.3 (APPS-Comp)')
ax.axvline(x=0.4, color='#fbbf24', linestyle='--', linewidth=1.6, alpha=0.75,
           label='τ*=0.4 (Omni-Math)')

# Regime labels
ax.text(0.055, 0.82, 'OVER-DENSE', color='#f87171', fontsize=8, alpha=0.9)
ax.text(0.055, 0.38, 'OPTIMAL RANGE', color='#4ade80', fontsize=8, alpha=0.9)
ax.text(0.055, 0.04, 'SPARSE', color='#fbbf24', fontsize=8, alpha=0.9)

ax.set_xlabel('Similarity Threshold τ', fontsize=10, color='white')
ax.set_ylabel('Avg Edge Density', fontsize=10, color='white')
ax.set_title('Avg Edge Density vs τ (code task, 4 agents)', fontsize=10, color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color(C_GRID)
ax.legend(facecolor='#1a2235', edgecolor=C_GRID, labelcolor='white', fontsize=8)
ax.grid(axis='y', color=C_GRID, alpha=0.35, zorder=0)
ax.set_ylim(0, 1.05)

# ── Right panel ────────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(PANEL_BG)

selected = {0.10: '#f87171', 0.20: '#fb923c', 0.30: '#38bdf8',
            0.40: '#4ade80', 0.50: '#a78bfa', 0.70: '#fbbf24', 0.90: '#94a3b8'}

for tau_sel, col in selected.items():
    key = min(tau_results.keys(), key=lambda k: abs(k - tau_sel))
    rd  = tau_results[key]
    r_labels = [f'R{r + 1}' for r, _ in rd]
    r_vals   = [d for _, d in rd]
    regime   = regime_color(np.mean(r_vals)).replace('#f87171', 'OD')
    ax2.plot(r_labels, r_vals, 'o-', color=col, linewidth=2.2, markersize=8,
             label=f'τ={key:.2f}')

ax2.set_xlabel('Round', fontsize=10, color='white')
ax2.set_ylabel('Edge Density', fontsize=10, color='white')
ax2.set_title('Per-Round Density for Selected τ values', fontsize=10, color='white')
ax2.tick_params(colors='white')
ax2.spines[:].set_color(C_GRID)
ax2.legend(facecolor='#1a2235', edgecolor=C_GRID, labelcolor='white', fontsize=8)
ax2.grid(color=C_GRID, alpha=0.35)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('/content/dytopo_tau_ablation_twopanel.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /content/dytopo_tau_ablation_twopanel.png')

In [ ]:
# Cell 7 — Combined overlay chart:
#   Primary y-axis:   paper accuracy curves (APPS-Comp + Omni-Math)
#   Secondary y-axis: structural edge density from mock sweep

BG       = "#0a0e1a"
PANEL_BG = "#0f1828"
C_GRID   = "#1e293b"
C_APPS   = "#38bdf8"
C_MATH   = "#fbbf24"
C_DENS   = "#94a3b8"

TAU_PAPER = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
APPS_COMP = [43.61, 44.84, 49.81, 47.35, 44.13, 40.86, 40.47, 37.59, 34.51]
OMNI_MATH = [42.86, 47.14, 50.00, 52.86, 48.57, 42.86, 41.43, 40.00, 38.58]

# Structural density on the paper tau grid (interpolated from sweep)
struct_taus  = np.array(sorted(tau_results.keys()))
struct_dens  = np.array([np.mean([d for _, d in tau_results[t]]) for t in struct_taus])
dens_at_paper_taus = np.interp(TAU_PAPER, struct_taus, struct_dens)

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL_BG)

ax2r = ax.twinx()

# ── Secondary axis: density ────────────────────────────────────────────────
ax2r.fill_between(struct_taus, struct_dens, alpha=0.13, color=C_DENS)
ax2r.plot(struct_taus, struct_dens, '--', color=C_DENS, linewidth=1.3,
          alpha=0.6, label='Edge density (structure, mock)')
ax2r.set_ylabel('Edge Density (mock structural sweep)', fontsize=9, color=C_DENS)
ax2r.tick_params(axis='y', colors=C_DENS)
ax2r.spines[:].set_color(C_GRID)
ax2r.set_ylim(0, 1.1)

# ── Primary axis: accuracy ─────────────────────────────────────────────────
x = np.array(TAU_PAPER)
ax.plot(x, APPS_COMP, 'o-', color=C_APPS, linewidth=2.5, markersize=8,
        label='APPS-Competition (τ*=0.3)', zorder=5)
ax.plot(x, OMNI_MATH, 's-', color=C_MATH, linewidth=2.5, markersize=8,
        label='Omni-Math (τ*=0.4)', zorder=5)

# Scatter overlay: density mapped to marker size on primary axis for visual correlation
ax.scatter(x, APPS_COMP, s=dens_at_paper_taus * 150 + 20,
           color=C_APPS, alpha=0.25, zorder=4)
ax.scatter(x, OMNI_MATH, s=dens_at_paper_taus * 150 + 20,
           color=C_MATH, alpha=0.25, zorder=4)

# Optimal region shading
ax.axvspan(0.28, 0.42, alpha=0.07, color='white', label='Optimal τ band (0.3–0.4)')
ax.axvline(x=0.3, color=C_APPS, linestyle='--', alpha=0.5, linewidth=1.4)
ax.axvline(x=0.4, color=C_MATH, linestyle='--', alpha=0.5, linewidth=1.4)

ax.annotate('49.81 %', xy=(0.3, 49.81), xytext=(0.33, 50.6),
            color=C_APPS, fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_APPS, lw=1.2))
ax.annotate('52.86 %', xy=(0.4, 52.86), xytext=(0.43, 53.6),
            color=C_MATH, fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_MATH, lw=1.2))

ax.set_xlabel('Similarity Threshold τ', fontsize=11, color='white')
ax.set_ylabel('Accuracy (%)', fontsize=11, color='white')
ax.set_title(
    'DyTopo: Paper Accuracy (Table 3) + Structural Edge Density (mock sweep)\n'
    'arXiv:2602.06039 — Lu et al. 2026',
    fontsize=10, color='white', pad=10)
ax.tick_params(colors='white')
ax.spines[:].set_color(C_GRID)
ax.set_xticks(x)
ax.set_ylim(30, 57)
ax.grid(color=C_GRID, alpha=0.35)

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2,
          facecolor='#1a2235', edgecolor=C_GRID, labelcolor='white', fontsize=8.5,
          loc='lower left')

plt.tight_layout()
plt.savefig('/content/dytopo_tau_overlay.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → /content/dytopo_tau_overlay.png')

In [ ]:
# Cell 8 — Summary DataFrame and key insight printout

TAU_PAPER = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
APPS_COMP = [43.61, 44.84, 49.81, 47.35, 44.13, 40.86, 40.47, 37.59, 34.51]
OMNI_MATH = [42.86, 47.14, 50.00, 52.86, 48.57, 42.86, 41.43, 40.00, 38.58]

struct_taus = np.array(sorted(tau_results.keys()))
struct_dens = np.array([np.mean([d for _, d in tau_results[t]]) for t in struct_taus])

rows = []
for tau, apps, omni in zip(TAU_PAPER, APPS_COMP, OMNI_MATH):
    # Average structural density at this tau (nearest key in sweep)
    nearest_tau = min(tau_results.keys(), key=lambda k: abs(k - tau))
    avg_density = np.mean([d for _, d in tau_results[nearest_tau]])
    per_round   = {r: d for r, d in tau_results[nearest_tau]}

    # Regime classification
    if avg_density > 0.70:
        regime = 'OVER-DENSE'
    elif avg_density < 0.10:
        regime = 'SPARSE'
    else:
        regime = 'OPTIMAL'

    rows.append({
        'tau'           : tau,
        'apps_acc_%'    : apps,
        'omni_acc_%'    : omni,
        'avg_density'   : round(avg_density, 3),
        'density_%'     : f"{avg_density * 100:.1f}",
        'round_1_dens'  : round(per_round.get(0, float('nan')), 3),
        'round_2_dens'  : round(per_round.get(1, float('nan')), 3),
        'round_3_dens'  : round(per_round.get(2, float('nan')), 3),
        'regime'        : regime,
        'apps_optimal'  : '★' if tau == 0.3 else '',
        'omni_optimal'  : '★' if tau == 0.4 else '',
    })

df = pd.DataFrame(rows)

# Display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

print('=' * 100)
print('  DyTopo τ Ablation Study — Summary Table')
print('  Paper: Lu et al. 2026 (arXiv:2602.06039)  |  Mock structural sweep, 4 agents, 3 rounds, code task')
print('=' * 100)
print(df.to_string(index=False))
print()

# Computed optima
best_apps_idx  = int(np.argmax(APPS_COMP))
best_omni_idx  = int(np.argmax(OMNI_MATH))
best_apps_tau  = TAU_PAPER[best_apps_idx]
best_omni_tau  = TAU_PAPER[best_omni_idx]
density_at_03  = df.loc[df['tau'] == 0.3, 'avg_density'].values[0]
density_at_04  = df.loc[df['tau'] == 0.4, 'avg_density'].values[0]

print('─' * 100)
print('KEY INSIGHTS')
print('─' * 100)
print(f'  1. APPS-Competition optimum  : τ = {best_apps_tau:.1f}  → {max(APPS_COMP):.2f} %  '
      f'(graph density at τ*: {density_at_03:.1%})')
print(f'  2. Omni-Math optimum         : τ = {best_omni_tau:.1f}  → {max(OMNI_MATH):.2f} %  '
      f'(graph density at τ*: {density_at_04:.1%})')
print()
print(f'  3. Inverted-U pattern: accuracy peaks at τ ∈ [0.3, 0.4] then monotonically declines.')
print(f'     - Δ accuracy (τ=0.3 vs τ=0.9) for APPS : {max(APPS_COMP) - APPS_COMP[-1]:+.2f} pp')
print(f'     - Δ accuracy (τ=0.4 vs τ=0.9) for Math  : {max(OMNI_MATH) - OMNI_MATH[-1]:+.2f} pp')
print()
print('  4. Structural correlation:')
print('     - τ < 0.3 → dense graphs allow all agents to communicate freely but drown signal in noise.')
print('     - τ ∈ [0.3, 0.4] → moderate density matches information-need: explore → verify → finalise.')
print('     - τ > 0.5 → sparse graphs starve agents of collaborative context needed to correct errors.')
print()
print('  5. Regime summary:')
for _, row in df.iterrows():
    star_apps = ' ← APPS optimal' if row['apps_optimal'] else ''
    star_omni = ' ← Omni-Math optimal' if row['omni_optimal'] else ''
    print(f'     τ={row["tau"]:.1f}  [{row["regime"]:10s}]  density={row["density_%"]:>5s} %'
          f'  APPS={row["apps_acc_%"]:.2f}%  Omni={row["omni_acc_%"]:.2f}%{star_apps}{star_omni}')
print('─' * 100)
print()
print('Saved plots:')
print('  /content/dytopo_table3_paper.png         — Table 3 bar+line reproduction')
print('  /content/dytopo_tau_ablation_twopanel.png — Two-panel structural analysis')
print('  /content/dytopo_tau_overlay.png           — Combined overlay (accuracy + density)')